# Chuẩn bị dữ liệu EEG và huấn luyện DL1 trên Google Drive

Chạy **Run All**. Notebook giữ kết quả trong `MyDrive/EEG-mini-project/results` và kết nối thư mục dữ liệu chia sẻ theo ID nếu cần. Lần đầu, Colab có thể yêu cầu chọn tài khoản Google để tạo lối tắt. Chọn cùng tài khoản đã mount Drive.

In [2]:
#@title Kết nối Drive và thư mục EEG được chia sẻ
from pathlib import Path
from time import monotonic, sleep

try:
    from google.colab import drive
except ImportError:
    drive = None
    drive_root = Path("G:/My Drive")
else:
    mount_point = Path("/content/drive")
    drive_root = mount_point / "MyDrive"
    if not drive_root.is_dir():
        drive.mount(str(mount_point))

DRIVE_PROJECT_DIR = drive_root / "EEG-mini-project"
DRIVE_DATA_FOLDER_ID = "14yFO5pINfs_-DET6nOAjZUKlz71FO62_"
DRIVE_DATA_SHORTCUT = drive_root / f"EEG-data-{DRIVE_DATA_FOLDER_ID}"
RAW_DIR = DRIVE_PROJECT_DIR / "ds004621_oddball"

def connect_shared_data():
    if DRIVE_DATA_SHORTCUT.is_dir():
        return DRIVE_DATA_SHORTCUT
    if drive is None:
        return None

    from google.colab import auth
    import google.auth
    from googleapiclient.discovery import build
    from googleapiclient.errors import HttpError

    auth.authenticate_user()
    credentials, _ = google.auth.default()
    service = build("drive", "v3", credentials=credentials, cache_discovery=False).files()
    try:
        target = service.get(
            fileId=DRIVE_DATA_FOLDER_ID, fields="name,mimeType", supportsAllDrives=True
        ).execute()
        if target["mimeType"] != "application/vnd.google-apps.folder":
            raise ValueError(f"Link dữ liệu phải trỏ tới thư mục Drive: {target['name']}")
        shortcuts = service.list(
            q=f"'root' in parents and name = '{DRIVE_DATA_SHORTCUT.name}' and trashed = false",
            fields="files(mimeType,shortcutDetails(targetId))", pageSize=100,
        ).execute()["files"]
        if any(item.get("shortcutDetails", {}).get("targetId") != DRIVE_DATA_FOLDER_ID for item in shortcuts):
            raise RuntimeError(f"Tên lối tắt đã được dùng cho mục khác: {DRIVE_DATA_SHORTCUT.name}")
        if not shortcuts:
            service.create(body={
                "name": DRIVE_DATA_SHORTCUT.name,
                "mimeType": "application/vnd.google-apps.shortcut",
                "shortcutDetails": {"targetId": DRIVE_DATA_FOLDER_ID},
            }, fields="id").execute()
    except HttpError as exc:
        raise RuntimeError(
            f"Drive API trả mã {exc.resp.status} khi mở hoặc tạo lối tắt cho link EEG. "
            "Kiểm tra tài khoản Colab có mở được link và có quyền tạo lối tắt trong My Drive."
        ) from exc

    deadline = monotonic() + 10
    while not DRIVE_DATA_SHORTCUT.is_dir() and monotonic() < deadline:
        sleep(1)
    if not DRIVE_DATA_SHORTCUT.is_dir():
        drive.mount(str(mount_point), force_remount=True)
    if not DRIVE_DATA_SHORTCUT.is_dir():
        raise FileNotFoundError(
            f"Colab chưa thấy lối tắt {DRIVE_DATA_SHORTCUT}. "
            "Hãy dùng cùng tài khoản Google khi mount Drive và khi xác thực link dữ liệu."
        )
    return DRIVE_DATA_SHORTCUT

print(f"Thư mục lưu kết quả: {DRIVE_PROJECT_DIR / 'results'}")


Mounted at /content/drive
Thư mục lưu kết quả: /content/drive/MyDrive/EEG-mini-project/results


## Chọn dữ liệu EEG

Notebook ưu tiên dữ liệu đã xử lý trong dự án. Nếu bộ dữ liệu chia sẻ đã có đủ đầu ra, notebook dùng bộ đó để train. Khi link chỉ có EEG gốc, pipeline đọc EEG từ link và lưu đầu ra vào `processed_khanh` trong dự án.

In [3]:
import json
import os
import sys

RUN_PREPROCESSING = True
EXPORT_FROM_CACHE = True  # Dùng lại toàn bộ cache của lần chạy trước
cwd = Path.cwd()
repo_dir = next((p for p in (cwd, *cwd.parents) if (p / "scripts/run_preprocessing.py").is_file()), None)
if repo_dir is None and "google.colab" in sys.modules:
    import subprocess
    checkout = Path("/content/EEG-mini-project")
    if not checkout.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "khanh",
                        "https://github.com/khanh186200/EEG-mini-project.git", str(checkout)], check=True)
    if (checkout / "scripts/run_preprocessing.py").is_file():
        repo_dir = checkout
if repo_dir is None:
    raise FileNotFoundError("Dùng %cd để vào repo EEG-mini-project rồi chạy lại ô này.")

if RUN_PREPROCESSING:
    processed_dir = DRIVE_PROJECT_DIR / "processed_khanh"
else:
    processed_dir = DRIVE_PROJECT_DIR / "processed"
    shortcut = processed_dir.with_suffix(".lnk")
    if os.name == "nt" and not processed_dir.is_dir() and shortcut.is_file():
        import subprocess

        # Drive for desktop hiển thị thư mục được chia sẻ dưới dạng lối tắt Windows.
        command = (
            "[Console]::OutputEncoding = [Text.UTF8Encoding]::new($false); "
            "(New-Object -ComObject WScript.Shell).CreateShortcut($env:EEG_SHORTCUT).TargetPath"
        )
        target = subprocess.check_output(
            ["powershell.exe", "-NoProfile", "-NonInteractive", "-Command", command],
            env={**os.environ, "EEG_SHORTCUT": str(shortcut)},
            encoding="utf-8",
        ).strip()
        if target:
            processed_dir = Path(target)
required_files = (
    "X_clean.npy", "X_no_ica_no_reject.npy", "metadata.csv", "metadata_no_ica.csv",
    "y.npy", "subject_ids.npy", "times.npy", "y_no_ica_no_reject.npy",
    "subject_ids_no_ica_no_reject.npy", "channel_names.json",
    "preprocessing_qc_summary.csv", "splits.json",
)

def has_output(path):
    return path.is_file() and path.stat().st_size > 0

def has_processed_data(directory):
    if not all(has_output(directory / filename) for filename in required_files):
        return False
    status_file = directory / "pipeline_status.json"
    if status_file.is_file():
        return json.loads(status_file.read_text(encoding="utf-8")).get("status") == "COMPLETED"
    return True

def has_raw_data(directory):
    return directory.is_dir() and next(directory.rglob("*.vhdr"), None) is not None

if not has_processed_data(processed_dir):
    shared_data = connect_shared_data()
    if shared_data is not None:
        print(f"Đã kết nối dữ liệu từ link Drive: {shared_data}")
        shared_processed = next((folder for folder in (
            shared_data / "processed_khanh", shared_data / "processed", shared_data,
        ) if has_processed_data(folder)), None)
        if shared_processed is not None:
            processed_dir = shared_processed
            RUN_PREPROCESSING = False
        elif RUN_PREPROCESSING and not has_raw_data(RAW_DIR):
            shared_raw = next((folder for folder in (
                shared_data / "ds004621_oddball", shared_data / "data/raw/oddball_data", shared_data,
            ) if has_raw_data(folder)), None)
            if shared_raw is not None:
                RAW_DIR = shared_raw
                print(f"Dữ liệu EEG gốc: {RAW_DIR}")

print(f"Thư mục dữ liệu đã xử lý được chọn: {processed_dir}")


Đã kết nối dữ liệu từ link Drive: /content/drive/MyDrive/EEG-data-14yFO5pINfs_-DET6nOAjZUKlz71FO62_
Thư mục dữ liệu đã xử lý được chọn: /content/drive/MyDrive/EEG-data-14yFO5pINfs_-DET6nOAjZUKlz71FO62_/processed


## Chạy pipeline V2.5 và tạo splits

Có đủ dữ liệu thì dùng lại. Nếu chỉ thiếu splits, notebook tạo splits từ metadata hiện có. Nếu cần khôi phục dữ liệu, notebook ưu tiên bộ cache đầy đủ khi `EXPORT_FROM_CACHE = True`; nếu thiếu cache, pipeline đọc EEG gốc và tiếp tục với `resume=True`.

In [5]:
def export_from_caches(output_dir, reference, generate_splits):
    import hashlib
    from datetime import datetime, timezone
    from tempfile import TemporaryDirectory

    import numpy as np
    import pandas as pd
    from src.preprocessing.pipeline import METADATA_COLUMNS

    subjects = reference["subjects"]
    paths = [output_dir / "per_subject" / f"{subject}_cache.npz" for subject in subjects]
    missing = [path.name for path in paths if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"Thiếu cache: {', '.join(missing)}")

    status_path = output_dir / "pipeline_status.json"
    status = json.loads(status_path.read_text(encoding="utf-8"))
    config = status["config"]
    times = np.linspace(config["EPOCHS"]["tmin"], config["EPOCHS"]["tmax"],
                        round((config["EPOCHS"]["tmax"] - config["EPOCHS"]["tmin"]) * config["RESAMPLE_FREQ"]) + 1)
    channels = None
    stats = []
    with TemporaryDirectory(prefix="export_", dir=output_dir) as folder:
        stage = Path(folder)
        for key, suffix, meta_name in [("clean", "", "metadata.csv"),
                                       ("no_ica", "_no_ica_no_reject", "metadata_no_ica.csv")]:
            labels, subject_ids, frames = [], [], []
            for subject, path in zip(subjects, paths):
                with np.load(path, allow_pickle=False) as cache:
                    names = cache["channel_names"].tolist()
                    if channels is None:
                        channels = names
                    if names != channels:
                        raise ValueError(f"{subject}: thứ tự kênh khác cache đầu tiên")
                    y = cache[f"y_{key}"]
                    columns = METADATA_COLUMNS + (["kept_in_clean"] if key == "no_ica" else [])
                    frame = pd.DataFrame(json.loads(cache[f"meta_{key}_json"].item()), columns=columns)
                    if y.dtype != np.int64 or not np.array_equal(y, frame["label"].to_numpy()) or not frame["subject_id"].eq(subject).all():
                        raise ValueError(f"{subject}: nhãn hoặc metadata {key} không khớp")
                    labels.append(y)
                    frames.append(frame)
                    subject_ids.append(np.full(len(y), subject, dtype="<U10"))
                    if key == "clean":
                        stats.append(json.loads(cache["stats_json"].item()))
            meta = pd.concat(frames, ignore_index=True)
            if key == "clean":
                digest = hashlib.sha256("\n".join(meta["trial_uid"]).encode()).hexdigest()
                if digest != reference["trial_uid_sha256"]:
                    raise ValueError("Thứ tự trial_uid khác chuẩn splits của nhóm")
            filename = "X_clean.npy" if key == "clean" else "X_no_ica_no_reject.npy"
            shape = (len(meta), len(channels), len(times))
            with (stage / filename).open("wb") as out:
                np.lib.format.write_array_header_2_0(out, {
                    "descr": np.dtype(np.float32).str, "fortran_order": False, "shape": shape,
                })
                for index, (path, y) in enumerate(zip(paths, labels), start=1):
                    with np.load(path, allow_pickle=False) as cache:
                        x = cache[f"X_{key}"]
                    if x.shape != (len(y), *shape[1:]) or x.dtype != np.float32:
                        raise ValueError(f"{path.name}: shape hoặc dtype của X_{key} không khớp")
                    out.write(memoryview(np.ascontiguousarray(x)).cast("B"))
                    del x
                    print(f"{filename}: {index}/{len(paths)} · {path.stem}", flush=True)
            np.save(stage / f"y{suffix}.npy", np.concatenate(labels))
            np.save(stage / f"subject_ids{suffix}.npy", np.concatenate(subject_ids))
            meta.to_csv(stage / meta_name, index=False)
        np.save(stage / "times.npy", times)
        (stage / "channel_names.json").write_text(json.dumps(channels), encoding="utf-8")
        (stage / "preprocessing_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
        for name in ("preprocessing_qc_summary.csv", "subject_summary.csv"):
            pd.DataFrame(stats).to_csv(stage / name, index=False)
        generate_splits(stage, seed=42)
        for path in sorted(stage.iterdir()):
            path.replace(output_dir / path.name)
        status.update(status="COMPLETED", successful_subjects=len(paths), failed_subjects=0,
                      end_time=datetime.now(timezone.utc).isoformat())
        manifest = stage / "pipeline_status.json"
        manifest.write_text(json.dumps(status, indent=2), encoding="utf-8")
        manifest.replace(status_path)

In [6]:
missing_data = [name for name in required_files if name != "splits.json"
                and not has_output(processed_dir / name)]
status_path = processed_dir / "pipeline_status.json"
status = json.loads(status_path.read_text(encoding="utf-8")) if status_path.is_file() else {}
needs_data = bool(missing_data) or status.get("status", "COMPLETED") != "COMPLETED"
needs_splits = not has_output(processed_dir / "splits.json")

if "google.colab" in sys.modules:
    from importlib.util import find_spec
    import subprocess

    packages = {"mne": "mne>=1.5.0", "numpy": "numpy", "pandas": "pandas",
                "scipy": "scipy", "sklearn": "scikit-learn", "tqdm": "tqdm", "pytest": "pytest"}
    missing_packages = [package for module, package in packages.items() if find_spec(module) is None]
    if missing_packages:
        subprocess.run([sys.executable, "-m", "pip", "install", *missing_packages], check=True)

if RUN_PREPROCESSING and (needs_data or needs_splits):
    if str(repo_dir) not in sys.path:
        sys.path.insert(0, str(repo_dir))
    from scripts.generate_splits import generate_splits

    if needs_data:
        reference = json.loads((repo_dir / "splits/loso.json").read_text(encoding="utf-8"))
        cache_dir = processed_dir / "per_subject"
        use_cache = EXPORT_FROM_CACHE and status_path.is_file() and all(
            has_output(cache_dir / f"{subject}_cache.npz") for subject in reference["subjects"]
        )
        if use_cache:
            print("Khôi phục đầu ra và splits từ cache trên Drive.")
            export_from_caches(processed_dir, reference, generate_splits)
        else:
            if not RAW_DIR.is_dir():
                raise FileNotFoundError(f"Thiếu đầu ra hoặc lượt chạy chưa hoàn tất. Kiểm tra dữ liệu gốc: {RAW_DIR}")
            from src.preprocessing.pipeline import CONFIG, run_pipeline
            config = {**CONFIG, "RAW_DIR": RAW_DIR, "PROCESSED_DIR": processed_dir,
                      "FIGURES_DIR": processed_dir / "figures" / "ica", "SAVE_SINGLE_SUBJECT_FIF": False}
            run_pipeline(config=config, resume=True)
            status = json.loads(status_path.read_text(encoding="utf-8"))
            if status["status"] != "COMPLETED":
                raise RuntimeError(f"Pipeline: {status['status']}. Xem {processed_dir / 'preprocessing.log'}")
            generate_splits(processed_dir, seed=42)
    else:
        print("Đã có dữ liệu V2.5. Tạo splits.json từ metadata.csv.")
        generate_splits(processed_dir, seed=42)
elif RUN_PREPROCESSING:
    print(f"Đã có dữ liệu V2.5 và splits.json trên Drive. Dùng lại {processed_dir}")

if not processed_dir.is_dir():
    raise FileNotFoundError(f"Thiếu thư mục dữ liệu đã xử lý: {processed_dir}")
os.environ["EEG_DATA_DIR"] = str(processed_dir.resolve())

In [7]:
missing = [name for name in required_files if not has_output(processed_dir / name)]
if missing:
    print("Các file còn thiếu cho tests/test_data_contract.py:")
    print("\n".join(missing))
else:
    print("Đã tìm thấy đủ 12 file đầu vào cho tests/test_data_contract.py.")

Đã tìm thấy đủ 12 file đầu vào cho tests/test_data_contract.py.


## Kiểm tra dữ liệu với pytest

Chạy ô dưới sau khi kết nối Drive. Nếu kernel thiếu thư viện, chạy `%pip install pytest mne numpy pandas` trong một ô riêng. Khi cả 5 kiểm tra đạt, pytest sẽ in `5 passed`.

Bộ test đọc đường dẫn từ `EEG_DATA_DIR`. Với script có tùy chọn `--data-dir`, truyền `"$processed_dir"` thay cho `data/processed/`. Dữ liệu được đọc trực tiếp từ Drive.

In [8]:
import subprocess
import sys

test_file = Path("tests/test_data_contract.py")
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-v", str(test_file)],
    cwd=repo_dir,
    env={**os.environ, "EEG_DATA_DIR": str(processed_dir.resolve()), "PYTHONIOENCODING": "utf-8"},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, encoding="utf-8", errors="replace",
)
print(result.stdout)
result.check_returncode()

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/EEG-mini-project
plugins: langsmith-0.14.17, anyio-4.16.0, typeguard-4.6.0, platformdirs-4.14.3
collecting ... collected 5 items

tests/test_data_contract.py::test_shapes_dtypes_and_excluded_subjects PASSED [ 20%]
tests/test_data_contract.py::test_channels_and_colin27_montage PASSED    [ 40%]
tests/test_data_contract.py::test_metadata_and_rt_validity PASSED        [ 60%]
tests/test_data_contract.py::test_splits_zero_leakage PASSED             [ 80%]
tests/test_data_contract.py::test_physiological_p300_sanity_check PASSED [100%]

======================== 5 passed in 149.34s (0:02:29) =========================



## Huấn luyện DL1 trên Google Drive

Sau khi dữ liệu đã được chuẩn bị, chạy các ô DL1 từ trên xuống: **Within → LOSO → CS5**. Notebook dùng mã đi kèm để các ô chạy cùng một phiên bản.

Trước mỗi phương thức, script kiểm tra báo cáo, dự đoán và cache trên Drive theo split của nhóm. Nếu đủ, ô dùng lại kết quả; nếu thiếu, ô train các fold còn thiếu. Chạy lại ô của phương thức đang dở để tiếp tục.

In [ ]:
#@title Mã DL1 đi kèm notebook
DL1_SNAPSHOT = {
    'scripts/train_dl1.py': '"""Huấn luyện DL1 (MS-CNN-BiGRU) theo thứ tự Within, LOSO và CS5."""\n\nimport argparse\nimport hashlib\nimport json\nimport shutil\nimport sys\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom tempfile import TemporaryDirectory\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nREPO_DIR = Path(__file__).resolve().parents[1]\nif str(REPO_DIR) not in sys.path:\n    sys.path.insert(0, str(REPO_DIR))\n\nfrom src.evaluation.metrics import (\n    METRIC_NAMES,\n    PREDICTION_COLUMNS,\n    aggregate_metrics,\n    save_metrics_json,\n    save_predictions_csv,\n)\nfrom src.training.dataset_dl1 import get_protocol_folds, load_eeg_dataset\n\nDL1_SOURCES = (\n    "src/models/ms_cnn_bigru.py",\n    "src/training/dataset_dl1.py",\n    "src/training/trainer_dl1.py",\n)\nSEED = 42\nPATIENCE = 15\n\n\n@dataclass(frozen=True)\nclass Experiment:\n    protocol: str\n    variant: str\n    label: str\n    name: str\n    fold_count: int\n    reference: str\n\n    def run_name(self, mode):\n        prefix = "pilot" if mode == "pilot" else "dl1"\n        return f"{prefix}_{self.name}"\n\n\nMETHODS = {\n    "within": (\n        Experiment("within_subject", "clean", "within", "ws_ms_cnn_bigru", 195, "within_5fold.json"),\n    ),\n    "loso": (\n        Experiment("cross_subject_loso", "clean", "loso", "loso_ms_cnn_bigru", 39, "loso.json"),\n    ),\n    "cs5": (\n        Experiment("cross_subject_5fold", "clean", "cs5_clean", "cs5_ms_cnn_bigru_clean", 5, "cross_5fold.json"),\n        Experiment("cross_subject_5fold", "no_ica", "cs5_no_ica", "cs5_ms_cnn_bigru_no_ica", 5, "cross_5fold.json"),\n    ),\n}\n\n\n@dataclass\nclass FoldResult:\n    metrics: dict\n    predictions: pd.DataFrame\n\n\ndef _experiment_config(experiment, bundle, mode):\n    return dict(\n        exp_name=experiment.run_name(mode), model="ms_cnn_bigru", model_kwargs={},\n        protocol=experiment.protocol, data_dir=str(bundle.data_dir), data_variant=bundle.data_variant,\n        epochs=1 if mode == "pilot" else 80, patience=PATIENCE, seed=SEED, lr=1e-3,\n        weight_decay=1e-2, batch_size=32 if experiment.protocol == "within_subject" else 64,\n        early_stop_metric="val_roc_auc", threshold_metric="balanced_accuracy",\n    )\n\n\ndef _load_folds(experiment, bundle, mode):\n    reference = REPO_DIR / "splits" / experiment.reference\n    protocol = experiment.protocol\n    # CS5 đã được kiểm tra trên clean và ánh xạ sang index của từng biến thể trong dataset loader.\n    if protocol != "cross_subject_5fold":\n        locked = json.loads(reference.read_text(encoding="utf-8"))\n        if locked["trial_uid_sha256"] != bundle.trial_uid_sha256 or locked[protocol] != bundle.splits[protocol]:\n            raise ValueError(f"{experiment.label}: dữ liệu hoặc fold khác chuẩn của nhóm")\n    folds = get_protocol_folds(bundle.splits, protocol)\n    if len(folds) != experiment.fold_count:\n        raise ValueError(f"{experiment.label}: cần {experiment.fold_count} fold, tìm được {len(folds)}")\n    if mode == "full":\n        return folds\n    if protocol == "cross_subject_5fold":\n        return [next(fold for fold in folds if fold["fold"] == 0)]\n    return folds[:1]\n\n\ndef _run_id(bundle, folds, config):\n    code_hash = hashlib.sha256()\n    for source in DL1_SOURCES:\n        code_hash.update((REPO_DIR / source).read_bytes())\n    if config["protocol"] == "cross_subject_5fold":\n        code_hash.update(json.dumps(folds, sort_keys=True).encode("utf-8"))\n    return f\'{bundle.trial_uid_sha256}:{code_hash.hexdigest()}:{config["epochs"]}:seed{SEED}\'\n\n\ndef _read_predictions(path):\n    return pd.read_csv(path, dtype={"trial_uid": str, "subject_id": str}, float_precision="round_trip")\n\n\ndef _validate_fold(metrics, predictions, bundle, fold):\n    metadata = ("fold_key", "protocol", "fold", "subject_id", "train_subjects", "val_subjects", "test_subjects")\n    if any(metrics.get(key) != fold[key] for key in metadata if key in fold):\n        raise ValueError("Metadata fold khác split đã khóa")\n    if not set(METRIC_NAMES).issubset(metrics) or not set(PREDICTION_COLUMNS).issubset(predictions):\n        raise ValueError("Thiếu cột metrics hoặc dự đoán")\n    test_idx = fold["test_idx"]\n    ids = bundle.metadata["trial_uid"].iloc[test_idx].astype(str)\n    indexed = predictions.set_index("trial_uid", verify_integrity=True)\n    if len(indexed) != len(ids) or not ids.isin(indexed.index).all():\n        raise ValueError("Danh sách trial test khác split đã khóa")\n    aligned = indexed.loc[ids]\n    if (not np.array_equal(aligned["y_true"], bundle.y[test_idx])\n            or not np.array_equal(aligned["subject_id"], bundle.subject_ids[test_idx])\n            or not aligned["protocol"].eq(fold["protocol"]).all()\n            or not aligned["fold"].eq(fold["fold"]).all()\n            or not aligned["y_prob"].between(0, 1).all()\n            or not aligned["threshold"].between(0, 1).all()\n            or not aligned["y_pred"].isin([0, 1]).all()):\n        raise ValueError("Dự đoán hoặc nhãn test không hợp lệ")\n    return FoldResult(metrics, aligned.reset_index())\n\n\ndef _read_experiment(directory, config, bundle, folds, run_id):\n    name = config["exp_name"]\n    metrics_path = directory / f"metrics_{name}.json"\n    predictions_path = directory / f"predictions_{name}.csv"\n    try:\n        report = json.loads(metrics_path.read_text(encoding="utf-8"))\n        saved_config = report["config"]\n        if (report["trial_uid_sha256"] != bundle.trial_uid_sha256\n                or any(saved_config.get(key) != value for key, value in config.items() if key != "data_dir")\n                or saved_config.get("run_id", run_id) != run_id):\n            raise ValueError("Cấu hình, dữ liệu hoặc mã huấn luyện đã thay đổi")\n        stored_metrics = {row["fold_key"]: row for row in report["per_fold"]}\n        predictions = _read_predictions(predictions_path)\n        group_columns = ["subject_id", "fold"] if config["protocol"] == "within_subject" else "fold"\n        stored_predictions = {group: rows for group, rows in predictions.groupby(group_columns, sort=False)}\n    except FileNotFoundError:\n        return {}\n    except (OSError, ValueError, KeyError, TypeError) as exc:\n        print(f"  Báo cáo trên Drive cần kiểm tra lại theo từng fold: {exc}", flush=True)\n        return {}\n\n    recovered = {}\n    for fold in folds:\n        key = fold["fold_key"]\n        group = (fold["subject_id"], fold["fold"]) if config["protocol"] == "within_subject" else fold["fold"]\n        try:\n            recovered[key] = _validate_fold(stored_metrics[key], stored_predictions[group], bundle, fold)\n        except (ValueError, KeyError, TypeError) as exc:\n            print(f"  Kết quả tổng hợp thiếu hoặc sai cho {key}: {exc}", flush=True)\n    return recovered\n\n\ndef _read_fold(cache, legacy, bundle, fold, run_id):\n    key = fold["fold_key"]\n    try:\n        record = json.loads((cache / f"{key}_metrics.json").read_text(encoding="utf-8"))\n        if record.pop("run_id", None) == run_id:\n            predictions = _read_predictions(cache / f"{key}_pred.csv")\n            return _validate_fold(record, predictions, bundle, fold)\n    except FileNotFoundError:\n        pass\n    except (OSError, ValueError, KeyError, TypeError) as exc:\n        print(f"  Cache {key} cần khôi phục: {exc}", flush=True)\n\n    try:\n        record = json.loads((legacy / f"{key}.json").read_text(encoding="utf-8"))\n        if record.get("run_id") == run_id:\n            metrics = dict(record["metrics"], best_epoch=record["best_epoch"], threshold=record["threshold"])\n            return _validate_fold(metrics, pd.DataFrame(record["predictions"]), bundle, fold)\n    except FileNotFoundError:\n        pass\n    except (OSError, ValueError, KeyError, TypeError) as exc:\n        print(f"  Kết quả cũ {key} cần khôi phục: {exc}", flush=True)\n    return None\n\n\ndef _write_fold(cache, key, result, run_id):\n    cache.mkdir(parents=True, exist_ok=True)\n    with TemporaryDirectory(dir=cache) as folder:\n        stage = Path(folder)\n        predictions = save_predictions_csv(result.predictions, stage / f"{key}_pred.csv")\n        metrics = stage / f"{key}_metrics.json"\n        metrics.write_text(json.dumps({**result.metrics, "run_id": run_id}), encoding="utf-8")\n        shutil.move(str(predictions), str(cache / predictions.name))\n        shutil.move(str(metrics), str(cache / metrics.name))\n\n\ndef _handover(results_dir, label, bundle, metrics, predictions):\n    ids = bundle.metadata["trial_uid"].astype(str)\n    indexed = predictions.set_index("trial_uid", verify_integrity=True)\n    if not ids.is_unique or len(indexed) != len(ids) or not ids.isin(indexed.index).all():\n        raise ValueError(f"{label}: dự đoán thiếu hoặc trùng trial")\n    aligned = indexed.loc[ids]\n    directory = results_dir / "metrics" / "dl1"\n    handover = results_dir / "handover_standard"\n    handover.mkdir(parents=True, exist_ok=True)\n    prefix = f"ms_cnn_bigru_{label}"\n    scores = pd.DataFrame(metrics).rename(columns={"balanced_accuracy": "bal_acc"})\n    columns = ["subject_id", "fold", "bal_acc", "roc_auc", "pr_auc", "f1_target", "sensitivity", "specificity"]\n    with TemporaryDirectory(dir=handover) as folder:\n        stage = Path(folder)\n        fold_file = stage / f"{prefix}_per_fold.csv"\n        pred_file = stage / f"{prefix}_preds.npz"\n        scores[columns].to_csv(fold_file, index=False)\n        np.savez_compressed(\n            pred_file, subject_id=aligned["subject_id"].to_numpy(dtype=str),\n            fold=aligned["fold"].to_numpy(dtype=np.int64), trial_idx=np.arange(len(ids), dtype=np.int64),\n            trial_uid=ids.to_numpy(dtype=str), y_true=aligned["y_true"].to_numpy(dtype=np.int64),\n            y_prob=aligned["y_prob"].to_numpy(dtype=np.float64), y_pred=aligned["y_pred"].to_numpy(dtype=np.int64),\n        )\n        csv_file = save_predictions_csv(aligned.reset_index(), stage / f"handover_predictions_dl1_{label}.csv")\n        shutil.move(str(csv_file), str(directory / csv_file.name))\n        for path in (fold_file, pred_file):\n            shutil.move(str(path), str(handover / path.name))\n\n\ndef _save_experiment(results_dir, experiment, config, bundle, results, mode, run_id):\n    directory = results_dir / "metrics" / "dl1"\n    directory.mkdir(parents=True, exist_ok=True)\n    name = config["exp_name"]\n    metrics = [result.metrics for result in results]\n    predictions = pd.concat([result.predictions for result in results], ignore_index=True)\n    agg = aggregate_metrics(metrics, experiment.protocol)\n    if mode == "full":\n        _handover(results_dir, experiment.label, bundle, metrics, predictions)\n    with TemporaryDirectory(dir=directory) as folder:\n        stage = Path(folder)\n        csv_file = save_predictions_csv(predictions, stage / f"predictions_{name}.csv")\n        json_file = save_metrics_json(\n            stage / f"metrics_{name}.json", {**config, "run_id": run_id}, bundle.trial_uid_sha256,\n            agg["summary"], metrics, agg["per_subject"],\n        )\n        for path in (csv_file, json_file):\n            shutil.move(str(path), str(directory / path.name))\n    print(f"  Kết quả: {directory / json_file.name}", flush=True)\n\n\ndef _restore_handover(results_dir, experiment, bundle, folds, results):\n    label = experiment.label\n    handover = results_dir / "handover_standard"\n    prefix = f"ms_cnn_bigru_{label}"\n    paths = (\n        results_dir / "metrics" / "dl1" / f"handover_predictions_dl1_{label}.csv",\n        handover / f"{prefix}_per_fold.csv",\n        handover / f"{prefix}_preds.npz",\n    )\n    if all(path.is_file() and path.stat().st_size for path in paths):\n        return\n    ordered = [results[fold["fold_key"]] for fold in folds]\n    metrics = [result.metrics for result in ordered]\n    predictions = pd.concat([result.predictions for result in ordered], ignore_index=True)\n    _handover(results_dir, label, bundle, metrics, predictions)\n\n\ndef _run_experiment(experiment, bundle, mode, results_dir):\n    folds = _load_folds(experiment, bundle, mode)\n    config = _experiment_config(experiment, bundle, mode)\n    run_id = _run_id(bundle, folds, config)\n    root = results_dir / "checkpoints" / "dl1" / config["exp_name"]\n    cache = root / "cache"\n    legacy = results_dir / "dl1" / mode / experiment.label\n    directory = results_dir / "metrics" / "dl1"\n    print(f"  Kiểm tra kết quả đã lưu: {experiment.label} ({len(folds)} fold)", flush=True)\n    results = _read_experiment(directory, config, bundle, folds, run_id)\n    if len(results) == len(folds):\n        print(f"  {experiment.label}: dùng lại đủ {len(folds)} fold đã lưu.", flush=True)\n        if mode == "full":\n            _restore_handover(results_dir, experiment, bundle, folds, results)\n        return\n\n    for fold in folds:\n        key = fold["fold_key"]\n        if key not in results:\n            cached = _read_fold(cache, legacy, bundle, fold, run_id)\n            if cached is not None:\n                results[key] = cached\n\n    pending = [fold for fold in folds if fold["fold_key"] not in results]\n    print(f"  {experiment.label}: dùng lại {len(results)}/{len(folds)} fold, cần train {len(pending)} fold.", flush=True)\n    if pending:\n        from src.training.trainer_dl1 import train_single_fold\n\n        checkpoint_dir = root if experiment.protocol != "within_subject" else None\n        for number, fold in enumerate(pending, start=1):\n            key = fold["fold_key"]\n            print(f"  [{number}/{len(pending)}] Train {experiment.label} {key}", flush=True)\n            trained = train_single_fold(\n                bundle, fold, epochs=config["epochs"], patience=PATIENCE, seed=SEED,\n                checkpoint_dir=checkpoint_dir,\n            )\n            metrics = dict(trained["fold_metrics"], best_epoch=trained["best_epoch"], threshold=trained["best_threshold"])\n            result = _validate_fold(metrics, trained["predictions_df"], bundle, fold)\n            _write_fold(cache, key, result, run_id)\n            results[key] = result\n            print(f"    ROC-AUC={metrics[\'roc_auc\']:.4f}, BA={metrics[\'balanced_accuracy\']:.4f}", flush=True)\n\n    _save_experiment(results_dir, experiment, config, bundle, [results[f["fold_key"]] for f in folds], mode, run_id)\n\n\ndef _save_subject_summary(results_dir, data_dir, trial_uid_sha256):\n    directory = results_dir / "metrics" / "dl1"\n    tables = []\n    for method in ("within", "loso"):\n        name = METHODS[method][0].run_name("full")\n        report_path = directory / f"metrics_{name}.json"\n        if not report_path.is_file():\n            return\n        try:\n            report = json.loads(report_path.read_text(encoding="utf-8"))\n            if report["trial_uid_sha256"] != trial_uid_sha256:\n                return\n            table = pd.DataFrame.from_dict(report["per_subject"], orient="index")[list(METRIC_NAMES)]\n        except (OSError, ValueError, KeyError, TypeError) as exc:\n            print(f"Chưa tổng hợp theo subject được từ {report_path.name}: {exc}", flush=True)\n            return\n        prefix = "ws" if method == "within" else "loso"\n        tables.append(table.add_prefix(f"{prefix}_"))\n    summary = pd.concat(tables, axis=1).rename_axis("subject_id")\n    qc = data_dir / "preprocessing_qc_summary.csv"\n    if qc.is_file():\n        summary = summary.join(pd.read_csv(qc).set_index("subject_id")[["n_clean_epochs_kept", "n_frequent", "n_target", "drop_rate_pct"]])\n    audit = REPO_DIR / "reports" / "audit_per_subject.csv"\n    if audit.is_file():\n        summary = summary.join(pd.read_csv(audit).set_index("subject_id")[["hit_rate_pct"]])\n    summary.to_csv(directory / "handover_per_subject_summary_dl1.csv")\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--data-dir", type=Path, required=True, help="Thư mục EEG đã xử lý")\n    parser.add_argument("--results-dir", type=Path, required=True, help="Thư mục lưu kết quả")\n    parser.add_argument("--method", choices=("all", *METHODS), default="all",\n                        help="Mặc định chạy lần lượt Within, LOSO và CS5")\n    parser.add_argument("--mode", choices=("full", "pilot", "cs5-full", "cs5-pilot"), default="full")\n    args = parser.parse_args(argv)\n    method, mode = args.method, args.mode\n    if mode.startswith("cs5-"):\n        if method not in ("all", "cs5"):\n            parser.error("Chế độ cs5-full/cs5-pilot chỉ dùng với phương thức CS5.")\n        method, mode = "cs5", mode.removeprefix("cs5-")\n    methods = list(METHODS) if method == "all" else [method]\n    device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"\n    print(f"DL1 MS-CNN-BiGRU | {mode.upper()} | {device}", flush=True)\n    print(f"Kết quả: {args.results_dir}", flush=True)\n    clean_hash = None\n    for step, name in enumerate(methods, start=1):\n        print(f"\\n[{step}/{len(methods)}] {name.upper()}", flush=True)\n        for experiment in METHODS[name]:\n            cs5_path = REPO_DIR / "splits" / experiment.reference if name == "cs5" else None\n            bundle = load_eeg_dataset(args.data_dir, data_variant=experiment.variant, cs5_path=cs5_path)\n            _run_experiment(experiment, bundle, mode, args.results_dir)\n            if name in ("within", "loso"):\n                clean_hash = bundle.trial_uid_sha256\n            del bundle\n    if mode == "full" and clean_hash is not None:\n        _save_subject_summary(args.results_dir, args.data_dir, clean_hash)\n    print("\\nHoàn tất DL1.", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n',
    'src/models/ms_cnn_bigru.py': '"""MS-CNN-BiGRU cho phân loại EEG P300 trên từng trial."""\n\nfrom collections import OrderedDict\n\nimport torch\nfrom torch import nn\n\n\nclass MSCNNBiGRU(nn.Module):\n    """Nhận (B, 127, T), trả raw logits (B, 1) cho BCEWithLogitsLoss.\n\n    Khi return_attention=True, trả thêm trọng số thời gian (B, T // 8).\n    """\n\n    def __init__(self) -> None:\n        super().__init__()\n        self.branches = nn.ModuleList(\n            nn.Sequential(OrderedDict(\n                temporal_conv=nn.Conv2d(1, 6, (1, k), padding=(0, k // 2), bias=False),\n                temporal_bn=nn.BatchNorm2d(6),\n                spatial_conv=nn.Conv2d(6, 12, (127, 1), groups=6, bias=False),\n                spatial_bn=nn.BatchNorm2d(12),\n                activation=nn.ELU(),\n                pool=nn.AvgPool2d((1, 4)),\n                dropout=nn.Dropout(0.25),\n            ))\n            for k in (125, 63, 31)\n        )\n        self.fusion = nn.Sequential(\n            nn.Conv1d(36, 36, 15, padding=7, groups=36, bias=False),\n            nn.Conv1d(36, 32, 1, bias=False),\n            nn.BatchNorm1d(32),\n            nn.ELU(),\n            nn.AvgPool1d(2),\n            nn.Dropout(0.30),\n        )\n        self.bigru = nn.GRU(32, 24, num_layers=2, batch_first=True,\n                            dropout=0.30, bidirectional=True)\n        self.layer_norm = nn.LayerNorm(48)\n        self.attention = nn.Sequential(nn.Linear(48, 24), nn.Tanh(), nn.Linear(24, 1, bias=False))\n        self.dropout = nn.Dropout(0.30)\n        self.classifier = nn.Linear(48, 1)\n\n    @torch.no_grad()\n    def apply_max_norm(self) -> None:\n        """Giới hạn chuẩn từng bộ lọc không gian và hàng trọng số phân loại."""\n        for branch in self.branches:\n            weight = branch.spatial_conv.weight\n            norms = weight.norm(p=2, dim=(1, 2, 3), keepdim=True)\n            weight.mul_((2.0 / (norms + 1e-8)).clamp(max=1.0))\n        weight = self.classifier.weight\n        weight.mul_((0.5 / (weight.norm(dim=1, keepdim=True) + 1e-8)).clamp(max=1.0))\n\n    def forward(\n        self, x: torch.Tensor, return_attention: bool = False\n    ) -> torch.Tensor | tuple[torch.Tensor, torch.Tensor]:\n        if x.ndim != 3 or x.shape[1] != 127 or x.shape[2] < 8:\n            raise ValueError(f"Đầu vào phải có shape (B, 127, T) với T >= 8, nhận {tuple(x.shape)}")\n\n        eeg = x.unsqueeze(1)\n        features = torch.cat([branch(eeg).squeeze(2) for branch in self.branches], dim=1)\n        sequence = self.fusion(features).transpose(1, 2)\n        if sequence.is_cuda and not self.training and sequence.requires_grad:\n            # Captum cần backward qua GRU trong eval; cuDNN chỉ hỗ trợ RNN backward khi train.\n            with torch.backends.cudnn.flags(enabled=False):\n                states, _ = self.bigru(sequence)\n        else:\n            states, _ = self.bigru(sequence)\n\n        states = self.layer_norm(states)\n        alpha = self.attention(states).squeeze(-1).softmax(dim=1)\n        pooled = torch.bmm(alpha.unsqueeze(1), states).squeeze(1)\n        logits = self.classifier(self.dropout(pooled))\n        if return_attention:\n            return logits, alpha\n        return logits\n',
    'src/training/dataset_dl1.py': '"""Tải dữ liệu chuẩn Data Contract và chuẩn hóa Zero-Leakage cho nhánh DL1."""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport re\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional, Sequence, Tuple, Union\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit\nfrom torch.utils.data import Dataset\n\nVALID_DATA_VARIANTS = ("clean", "no_ica")\nVALID_PROTOCOLS = ("within_subject", "cross_subject_5fold", "cross_subject_loso")\n\n\ndef natural_sort_key(value: str) -> List[Union[int, str]]:\n    """Khóa sắp xếp tự nhiên theo số học (vd: sub-02 trước sub-10)."""\n    return [int(tok) if tok.isdigit() else tok.lower() for tok in re.split(r"(\\d+)", str(value))]\n\n\ndef compute_trial_uid_sha256(trial_uids: Sequence[str]) -> str:\n    """Tính mã băm SHA-256 của danh sách `trial_uid` theo đúng chuẩn Data Contract."""\n    payload = "\\n".join(str(uid) for uid in trial_uids).encode("utf-8")\n    return hashlib.sha256(payload).hexdigest()\n\n\n@dataclass\nclass EEGDatasetBundle:\n    """Gói dữ liệu đầu vào đã được xác thực hợp đồng dữ liệu."""\n\n    data_dir: Path\n    data_variant: str\n    X: np.ndarray\n    y: np.ndarray\n    subject_ids: np.ndarray\n    metadata: pd.DataFrame\n    splits: Dict[str, Any]\n    trial_uid_sha256: str\n\n\nclass ChannelStandardScaler:\n    """Bộ chuẩn hóa Z-score theo từng kênh EEG với cơ chế chống sập đơn vị Volts.\n\n    Dữ liệu EEG thô lưu theo chuẩn MNE ở đơn vị Volts (độ lệch chuẩn ~1e-5 V).\n    Nếu dùng trực tiếp `eps=1e-5` ở đơn vị Volts, mẫu số `(std + eps)` sẽ bóp méo\n    toàn bộ biên độ tín hiệu. Lớp này quy đổi nội bộ sang microvolts (`* 1e6`) trên\n    độ chính xác kép `float64` với `eps=1e-8`, sau đó trả về tensor `float32`.\n    """\n\n    def __init__(self, eps: float = 1e-8, scale_factor: float = 1e6) -> None:\n        if eps <= 0.0:\n            raise ValueError(f"eps phải > 0, nhận được {eps}.")\n        if scale_factor <= 0.0:\n            raise ValueError(f"scale_factor phải > 0, nhận được {scale_factor}.")\n        self.eps = float(eps)\n        self.scale_factor = float(scale_factor)\n        self.mean_: Optional[np.ndarray] = None\n        self.std_: Optional[np.ndarray] = None\n        self.n_channels_: Optional[int] = None\n\n    @property\n    def is_fitted(self) -> bool:\n        return self.mean_ is not None and self.std_ is not None and self.n_channels_ is not None\n\n    def fit(self, X_train: np.ndarray) -> "ChannelStandardScaler":\n        """Chỉ ước lượng mean và std theo kênh từ tập huấn luyện `X_train` `(N, C, T)`."""\n        arr = np.asarray(X_train)\n        if arr.ndim != 3:\n            raise ValueError(f"Kỳ vọng tensor 3D (N, C, T), nhận được shape {arr.shape}.")\n        if arr.shape[0] == 0 or arr.shape[2] == 0:\n            raise ValueError(f"Tập X_train rỗng không thể fit scaler: shape={arr.shape}.")\n\n        x_uv = arr.astype(np.float64) * self.scale_factor\n        mean = np.mean(x_uv, axis=(0, 2), keepdims=True)\n        std = np.std(x_uv, axis=(0, 2), keepdims=True)\n\n        if not (np.isfinite(mean).all() and np.isfinite(std).all()):\n            raise ValueError("Phát hiện giá trị NaN/Inf trong X_train khi fit ChannelStandardScaler.")\n\n        self.mean_ = mean\n        self.std_ = std\n        self.n_channels_ = int(arr.shape[1])\n        return self\n\n    def transform(self, X: np.ndarray) -> np.ndarray:\n        """Chuẩn hóa `X` bằng đúng thống kê đã fit trên train (tuyệt đối không refit)."""\n        if not self.is_fitted:\n            raise RuntimeError("ChannelStandardScaler chưa được fit trước khi gọi transform().")\n\n        arr = np.asarray(X)\n        if arr.ndim != 3:\n            raise ValueError(f"Kỳ vọng tensor 3D (N, C, T), nhận được shape {arr.shape}.")\n        if arr.shape[1] != self.n_channels_:\n            raise ValueError(\n                f"Số kênh không khớp scaler đã fit ({self.n_channels_}), nhận được {arr.shape[1]}."\n            )\n\n        assert self.mean_ is not None and self.std_ is not None\n        x_uv = arr.astype(np.float64) * self.scale_factor\n        normed = (x_uv - self.mean_) / (self.std_ + self.eps)\n        return normed.astype(np.float32)\n\n    def fit_transform(self, X_train: np.ndarray) -> np.ndarray:\n        """Fit trên `X_train` và trả về `X_train` đã chuẩn hóa `float32`."""\n        return self.fit(X_train).transform(X_train)\n\n    def state_dict(self) -> Dict[str, Any]:\n        """Xuất trạng thái scaler tương thích `torch.save(..., weights_only=True)`."""\n        if not self.is_fitted:\n            raise RuntimeError("Không thể xuất state_dict() khi scaler chưa được fit.")\n        assert self.mean_ is not None and self.std_ is not None and self.n_channels_ is not None\n        return {\n            "mean": torch.from_numpy(self.mean_.copy()).to(dtype=torch.float64),\n            "std": torch.from_numpy(self.std_.copy()).to(dtype=torch.float64),\n            "eps": float(self.eps),\n            "scale_factor": float(self.scale_factor),\n            "n_channels": int(self.n_channels_),\n        }\n\n    def load_state_dict(self, state: Dict[str, Any]) -> "ChannelStandardScaler":\n        """Nạp lại trạng thái scaler từ checkpoint `.pt`."""\n        for key in ("mean", "std", "eps", "scale_factor"):\n            if key not in state:\n                raise KeyError(f"state_dict thiếu khóa bắt buộc \'{key}\'.")\n\n        mean_raw = state["mean"]\n        std_raw = state["std"]\n        mean_arr = (\n            mean_raw.detach().cpu().numpy().astype(np.float64)\n            if isinstance(mean_raw, torch.Tensor)\n            else np.asarray(mean_raw, dtype=np.float64)\n        )\n        std_arr = (\n            std_raw.detach().cpu().numpy().astype(np.float64)\n            if isinstance(std_raw, torch.Tensor)\n            else np.asarray(std_raw, dtype=np.float64)\n        )\n\n        if mean_arr.ndim != 3 or std_arr.ndim != 3 or mean_arr.shape != std_arr.shape:\n            raise ValueError(\n                f"Shape mean/std không hợp lệ trong state_dict: {mean_arr.shape} vs {std_arr.shape}."\n            )\n\n        self.eps = float(state["eps"])\n        self.scale_factor = float(state["scale_factor"])\n        self.mean_ = mean_arr\n        self.std_ = std_arr\n        self.n_channels_ = int(state.get("n_channels", mean_arr.shape[1]))\n        return self\n\n    @classmethod\n    def from_state_dict(cls, state: Dict[str, Any]) -> "ChannelStandardScaler":\n        """Khởi tạo mới một `ChannelStandardScaler` từ `state_dict` đã lưu."""\n        scaler = cls(\n            eps=float(state["eps"]),\n            scale_factor=float(state["scale_factor"]),\n        )\n        return scaler.load_state_dict(state)\n\n\ndef _build_within_splits_for_arrays(\n    subjects: np.ndarray,\n    y: np.ndarray,\n    unique_subjects: List[str],\n    seed: int,\n) -> Dict[str, List[Dict[str, Any]]]:\n    """Sinh lại chia tập `within_subject` theo cùng seed khi dùng nhánh `no_ica`."""\n    within_splits: Dict[str, List[Dict[str, Any]]] = {}\n    for sub_id in unique_subjects:\n        sub_indices = np.where(subjects == sub_id)[0]\n        sub_y = y[sub_indices]\n        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)\n        sub_folds: List[Dict[str, Any]] = []\n        for fold_idx, (tr_val_pos, te_pos) in enumerate(skf.split(sub_indices, sub_y)):\n            train_val_idx = sub_indices[tr_val_pos]\n            test_idx = sub_indices[te_pos]\n            train_val_y = y[train_val_idx]\n\n            sss = StratifiedShuffleSplit(\n                n_splits=1, test_size=0.2, random_state=seed + fold_idx\n            )\n            tr_pos, val_pos = next(sss.split(train_val_idx, train_val_y))\n            sub_folds.append(\n                {\n                    "fold": int(fold_idx),\n                    "train_idx": np.sort(train_val_idx[tr_pos]).astype(int).tolist(),\n                    "val_idx": np.sort(train_val_idx[val_pos]).astype(int).tolist(),\n                    "test_idx": np.sort(test_idx).astype(int).tolist(),\n                }\n            )\n        within_splits[sub_id] = sub_folds\n    return within_splits\n\n\ndef _map_subject_folds_to_indices(\n    clean_folds: Sequence[Dict[str, Any]],\n    subjects_target: np.ndarray,\n) -> List[Dict[str, Any]]:\n    """Ánh xạ danh sách subject theo từng fold từ `splits.json` sang chỉ số mẫu thực tế."""\n    mapped_folds: List[Dict[str, Any]] = []\n    available_subjects = set(np.unique(subjects_target).tolist())\n\n    for fold in clean_folds:\n        train_subjects = [str(s) for s in fold["train_subjects"]]\n        val_subjects = [str(s) for s in fold["val_subjects"]]\n        test_subjects = [str(s) for s in fold["test_subjects"]]\n\n        expected_subjects = set(train_subjects) | set(val_subjects) | set(test_subjects)\n        missing = expected_subjects - available_subjects\n        if missing:\n            raise ValueError(\n                f"Nhánh no_ica thiếu các subject có trong splits.json: {sorted(missing)}"\n            )\n\n        train_idx = np.where(np.isin(subjects_target, train_subjects))[0]\n        val_idx = np.where(np.isin(subjects_target, val_subjects))[0]\n        test_idx = np.where(np.isin(subjects_target, test_subjects))[0]\n\n        if len(train_idx) == 0 or len(val_idx) == 0 or len(test_idx) == 0:\n            raise ValueError(\n                f"Fold {fold.get(\'fold\')} của no_ica có tập rỗng "\n                f"(train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)})."\n            )\n\n        mapped_folds.append(\n            {\n                "fold": int(fold["fold"]),\n                "train_subjects": train_subjects,\n                "val_subjects": val_subjects,\n                "test_subjects": test_subjects,\n                "train_idx": np.sort(train_idx).astype(int).tolist(),\n                "val_idx": np.sort(val_idx).astype(int).tolist(),\n                "test_idx": np.sort(test_idx).astype(int).tolist(),\n            }\n        )\n    return mapped_folds\n\n\ndef _build_no_ica_splits(\n    clean_splits: Dict[str, Any],\n    subjects_no_ica: np.ndarray,\n    y_no_ica: np.ndarray,\n    no_ica_sha256: str,\n) -> Dict[str, Any]:\n    """Tạo bộ splits đồng bộ subject với `splits.json` cho biến thể `no_ica`."""\n    seed = int(clean_splits.get("seed", 42))\n    unique_subjects = sorted(np.unique(subjects_no_ica).tolist(), key=natural_sort_key)\n\n    return {\n        "schema_version": clean_splits.get("schema_version", "2.5.0"),\n        "seed": seed,\n        "n_samples": int(len(subjects_no_ica)),\n        "n_subjects": int(len(unique_subjects)),\n        "subjects": unique_subjects,\n        "trial_uid_sha256": no_ica_sha256,\n        "clean_trial_uid_sha256": clean_splits.get("trial_uid_sha256"),\n        "within_subject": _build_within_splits_for_arrays(\n            subjects_no_ica, y_no_ica, unique_subjects, seed\n        ),\n        "cross_subject_5fold": _map_subject_folds_to_indices(\n            clean_splits.get("cross_subject_5fold", []), subjects_no_ica\n        ),\n        "cross_subject_loso": _map_subject_folds_to_indices(\n            clean_splits.get("cross_subject_loso", []), subjects_no_ica\n        ),\n    }\n\n\ndef _validate_loaded_arrays(\n    X: np.ndarray,\n    y: np.ndarray,\n    subject_ids: np.ndarray,\n    metadata: pd.DataFrame,\n    variant: str,\n) -> None:\n    """Xác thực shape, kiểu dữ liệu và tính nhất quán giữa mảng numpy và metadata."""\n    if X.ndim != 3 or X.shape[1:] != (127, 251):\n        raise ValueError(\n            f"[{variant}] X phải có shape (N, 127, 251), nhận được {X.shape}."\n        )\n    n_samples = X.shape[0]\n    if y.shape != (n_samples,):\n        raise ValueError(f"[{variant}] y có shape {y.shape}, kỳ vọng ({n_samples},).")\n    if subject_ids.shape != (n_samples,):\n        raise ValueError(\n            f"[{variant}] subject_ids có shape {subject_ids.shape}, kỳ vọng ({n_samples},)."\n        )\n    if len(metadata) != n_samples:\n        raise ValueError(\n            f"[{variant}] metadata có {len(metadata)} dòng, không khớp X ({n_samples})."\n        )\n    for required_col in ("trial_uid", "subject_id", "label"):\n        if required_col not in metadata.columns:\n            raise ValueError(f"[{variant}] metadata thiếu cột bắt buộc \'{required_col}\'.")\n    if not all(np.isfinite(X[start:start + 64]).all() for start in range(0, len(X), 64)):\n        raise ValueError(f"[{variant}] Dữ liệu X chứa NaN hoặc Inf.")\n    if not np.array_equal(metadata["label"].to_numpy(dtype=np.int64), y):\n        raise ValueError(f"[{variant}] Cột \'label\' trong metadata không khớp y.npy.")\n    if not np.array_equal(metadata["subject_id"].astype(str).to_numpy(), subject_ids):\n        raise ValueError(\n            f"[{variant}] Cột \'subject_id\' trong metadata không khớp subject_ids.npy."\n        )\n\n\ndef _locked_cs5(path: Path, clean_metadata: pd.DataFrame, clean_hash: str, seed: int) -> list[dict]:\n    """Kiểm tra cohort, thứ tự trial và từng index trước khi dùng CS5 của nhóm."""\n    ref = json.loads(path.read_text(encoding="utf-8"))\n    digest = compute_trial_uid_sha256(clean_metadata["trial_uid"])\n    if (ref["protocol"] != "cross_subject_5fold" or ref["seed"] != seed\n            or ref["trial_uid_sha256"] != digest or digest != clean_hash):\n        raise ValueError("CS5 tham chiếu không khớp protocol, seed hoặc hash của dữ liệu clean")\n    subjects = clean_metadata["subject_id"].astype(str).to_numpy()\n    folds = ref["cross_subject_5fold"]\n    if len(folds) != 5 or {f["fold"] for f in folds} != set(range(5)):\n        raise ValueError("CS5 tham chiếu phải có đủ fold 0–4")\n    cohort = set(subjects)\n    tested = set()\n    for fold in folds:\n        assigned = set()\n        for part in ("train", "val", "test"):\n            group = set(fold[f"{part}_subjects"])\n            indices = fold[f"{part}_idx"]\n            expected = np.flatnonzero(np.isin(subjects, list(group)))\n            if (not group or not group <= cohort or assigned & group\n                    or not np.array_equal(np.sort(indices), expected)):\n                raise ValueError(f"CS5 fold {fold[\'fold\']}: subject hoặc index {part} không khớp")\n            assigned.update(group)\n        test = set(fold["test_subjects"])\n        if assigned != cohort or tested & test:\n            raise ValueError(f"CS5 fold {fold[\'fold\']}: thiếu subject hoặc lặp subject test")\n        tested.update(test)\n    if tested != cohort:\n        raise ValueError("CS5 thiếu subject trong các tập test")\n    return folds\n\n\ndef load_eeg_dataset(data_dir: str | Path, data_variant: str = "clean", *,\n                     early_450: bool = False, cs5_path: str | Path | None = None) -> EEGDatasetBundle:\n    """Đọc EEG; truyền cs5_path để dùng phân nhóm CS5 đã khóa cho cả clean và no_ica."""\n    if data_variant not in VALID_DATA_VARIANTS:\n        raise ValueError(f"Biến thể dữ liệu không hợp lệ: {data_variant}")\n    root = Path(data_dir)\n    clean = data_variant == "clean"\n    suffix = "" if clean else "_no_ica_no_reject"\n    x_name = "X_clean.npy" if clean else "X_no_ica_no_reject.npy"\n    X = np.load(root / x_name, mmap_mode="r").astype(np.float32, copy=False)\n    y = np.load(root / f"y{suffix}.npy").astype(np.int64, copy=False)\n    subject_ids = np.load(root / f"subject_ids{suffix}.npy").astype(str)\n    metadata = pd.read_csv(root / ("metadata.csv" if clean else "metadata_no_ica.csv"))\n    _validate_loaded_arrays(X, y, subject_ids, metadata, variant=data_variant)\n    splits = json.loads((root / "splits.json").read_text(encoding="utf-8"))\n    digest = compute_trial_uid_sha256(metadata["trial_uid"].astype(str))\n    if cs5_path is not None:\n        clean_meta = metadata if clean else pd.read_csv(root / "metadata.csv", usecols=["trial_uid", "subject_id"])\n        if not clean and set(subject_ids) != set(clean_meta["subject_id"]):\n            raise ValueError("CS5 yêu cầu cùng cohort subject giữa clean và no_ica")\n        splits["cross_subject_5fold"] = _locked_cs5(\n            Path(cs5_path), clean_meta, splits["trial_uid_sha256"], splits["seed"]\n        )\n    if clean:\n        if digest != splits["trial_uid_sha256"]:\n            raise ValueError("SHA-256 của trial_uid không khớp splits.json")\n    else:\n        splits = _build_no_ica_splits(splits, subject_ids, y, digest)\n    return EEGDatasetBundle(root, data_variant, X[:, :, :163] if early_450 else X,\n                            y, subject_ids, metadata, splits, digest)\n\n\ndef get_protocol_folds(\n    splits: Dict[str, Any],\n    protocol: str,\n    subjects: Optional[Sequence[str]] = None,\n    folds: Optional[Sequence[int]] = None,\n) -> List[Dict[str, Any]]:\n    """Trích xuất danh sách fold chuẩn hóa cho một giao thức đánh giá.\n\n    Mỗi phần tử trả về có khóa:\n        `fold_key`, `protocol`, `fold`, `subject_id`, `train_idx`, `val_idx`, `test_idx`,\n        và tùy chọn `train_subjects`, `val_subjects`, `test_subjects`.\n    """\n    if protocol not in VALID_PROTOCOLS:\n        raise ValueError(f"protocol phải thuộc {VALID_PROTOCOLS}, nhận được {protocol!r}.")\n\n    subject_filter = set(str(s) for s in subjects) if subjects else None\n    fold_filter = set(int(f) for f in folds) if folds is not None else None\n    specs: List[Dict[str, Any]] = []\n\n    if protocol == "within_subject":\n        within_dict = splits.get("within_subject", {})\n        ordered_subs = sorted(within_dict.keys(), key=natural_sort_key)\n        for sub_id in ordered_subs:\n            if subject_filter is not None and sub_id not in subject_filter:\n                continue\n            for item in within_dict[sub_id]:\n                fold_idx = int(item["fold"])\n                if fold_filter is not None and fold_idx not in fold_filter:\n                    continue\n                specs.append(\n                    {\n                        "fold_key": f"{sub_id}_fold{fold_idx}",\n                        "protocol": protocol,\n                        "fold": fold_idx,\n                        "subject_id": sub_id,\n                        "train_idx": item["train_idx"],\n                        "val_idx": item["val_idx"],\n                        "test_idx": item["test_idx"],\n                    }\n                )\n        return specs\n\n    raw_folds = splits.get(protocol, [])\n    for item in raw_folds:\n        fold_idx = int(item["fold"])\n        if fold_filter is not None and fold_idx not in fold_filter:\n            continue\n        test_subs = [str(s) for s in item.get("test_subjects", [])]\n        if subject_filter is not None and not (set(test_subs) & subject_filter):\n            continue\n\n        if protocol == "cross_subject_loso" and len(test_subs) == 1:\n            sub_label = test_subs[0]\n            fold_key = f"{sub_label}_fold{fold_idx}"\n        else:\n            sub_label = ",".join(test_subs) if test_subs else "all"\n            fold_key = f"fold{fold_idx}"\n\n        specs.append(\n            {\n                "fold_key": fold_key,\n                "protocol": protocol,\n                "fold": fold_idx,\n                "subject_id": sub_label,\n                "train_subjects": [str(s) for s in item.get("train_subjects", [])],\n                "val_subjects": [str(s) for s in item.get("val_subjects", [])],\n                "test_subjects": test_subs,\n                "train_idx": item["train_idx"],\n                "val_idx": item["val_idx"],\n                "test_idx": item["test_idx"],\n            }\n        )\n    return specs\n\n\ndef fit_fold_scaler(x: np.ndarray, train_idx: np.ndarray) -> ChannelStandardScaler:\n    """Gộp mean/variance từng lô train bằng float64, tránh bản sao float64 của cả fold LOSO."""\n    scaler = ChannelStandardScaler()\n    mean = np.zeros((1, x.shape[1], 1), dtype=np.float64)\n    m2 = np.zeros_like(mean)\n    count = 0\n    for start in range(0, len(train_idx), 64):\n        block = x[train_idx[start:start + 64]].astype(np.float64)\n        block *= scaler.scale_factor\n        n = len(block) * block.shape[2]\n        block_mean = block.mean(axis=(0, 2), keepdims=True)\n        delta = block_mean - mean\n        block -= block_mean\n        m2 += np.square(block, out=block).sum(axis=(0, 2), keepdims=True)\n        m2 += delta**2 * (count * n / (count + n))\n        mean += delta * (n / (count + n))\n        count += n\n    if not count or not np.isfinite(mean).all() or not np.isfinite(m2).all():\n        raise ValueError("Tập train rỗng hoặc chứa NaN/Inf")\n    scaler.mean_, scaler.std_, scaler.n_channels_ = mean, np.sqrt(m2 / count), x.shape[1]\n    return scaler\n\n\nclass FoldDataset(Dataset):\n    """Chuẩn hóa trial khi lấy mẫu, giữ thứ tự index của split cho validation và test."""\n\n    def __init__(self, bundle: EEGDatasetBundle, indices: np.ndarray, scaler: ChannelStandardScaler):\n        self.x, self.y = bundle.X, bundle.y\n        self.indices, self.scaler = indices, scaler\n\n    def __len__(self):\n        return len(self.indices)\n\n    def __getitem__(self, item):\n        i = self.indices[item]\n        x = self.scaler.transform(self.x[i:i + 1])[0]\n        return torch.from_numpy(x), torch.tensor([self.y[i]], dtype=torch.float32)\n',
    'src/training/trainer_dl1.py': '"""Huấn luyện và đánh giá MS-CNN-BiGRU trên một fold đã khóa."""\n\nimport random\nimport shutil\nimport time\nfrom pathlib import Path\nfrom tempfile import NamedTemporaryFile\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader\n\nfrom src.evaluation.metrics import PREDICTION_COLUMNS, compute_binary_metrics, find_optimal_threshold\nfrom src.models.ms_cnn_bigru import MSCNNBiGRU\nfrom .dataset_dl1 import (\n    VALID_PROTOCOLS, ChannelStandardScaler, EEGDatasetBundle, FoldDataset, fit_fold_scaler,\n)\n\n\ndef _split_indices(bundle, fold):\n    """Chặn index sai, trial trùng và subject bị rò rỉ trước khi fit scaler."""\n    protocol = fold["protocol"]\n    if protocol not in VALID_PROTOCOLS:\n        raise ValueError(f"Giao thức không hợp lệ: {protocol}")\n    parts = []\n    for name in ("train_idx", "val_idx", "test_idx"):\n        idx = np.asarray(fold[name])\n        if (idx.ndim != 1 or not idx.size or idx.dtype.kind not in "iu"\n                or idx.min() < 0 or idx.max() >= len(bundle.y)):\n            raise ValueError(f"{name} rỗng hoặc chứa index không hợp lệ")\n        parts.append(idx)\n    indices = np.concatenate(parts)\n    if len(np.unique(indices)) != len(indices):\n        raise ValueError("Có trial trùng trong fold")\n    subjects = [set(bundle.subject_ids[idx]) for idx in parts]\n    tr, va, te = subjects\n    if protocol == "within_subject":\n        if len(tr | va | te) != 1:\n            raise ValueError("Within-Subject chỉ được chứa một subject")\n    elif tr & va or tr & te or va & te or (protocol == "cross_subject_loso" and len(te) != 1):\n        raise ValueError("Fold cross-subject không tách subject đúng giao thức")\n    return parts\n\n\n@torch.inference_mode()\ndef _predict(model, loader, device):\n    model.eval()\n    return torch.cat([model(x.to(device)).sigmoid().flatten().cpu() for x, _ in loader]).numpy()\n\n\ndef load_checkpoint_for_eval(path, device="cpu"):\n    """Nạp model ở eval mode, scaler và ngưỡng trong checkpoint DL1."""\n    ckpt = torch.load(path, map_location="cpu", weights_only=True)\n    if ckpt["model_name"] != "ms_cnn_bigru":\n        raise ValueError(f"Checkpoint không phải DL1: {ckpt[\'model_name\']}")\n    model = MSCNNBiGRU(**ckpt["model_kwargs"])\n    model.load_state_dict(ckpt["model_state_dict"])\n    return model.to(device).eval(), ChannelStandardScaler.from_state_dict(ckpt["scaler_state_dict"]), ckpt\n\n\ndef train_single_fold(\n    bundle: EEGDatasetBundle, fold: dict, *, epochs: int = 80, patience: int = 15,\n    batch_size: int | None = None, seed: int = 42, device=None,\n    checkpoint_dir: str | Path | None = None,\n) -> dict:\n    """Nhận một fold từ get_protocol_folds; truyền checkpoint_dir để lưu file .pt."""\n    tr, va, te = _split_indices(bundle, fold)\n    if batch_size is None:\n        batch_size = 32 if fold["protocol"] == "within_subject" else 64\n    if min(epochs, patience, batch_size) < 1:\n        raise ValueError("epochs, patience và batch_size phải dương")\n    counts = np.bincount(bundle.y[tr], minlength=2)\n    if len(counts) != 2 or min(counts) == 0:\n        raise ValueError("Tập train phải có đủ nhãn 0 và 1")\n    pos_weight = float(counts[0] / counts[1])\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n    device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))\n    start = time.perf_counter()\n    print(f"  [{fold[\'fold_key\']}] Đang chuẩn hóa kênh tín hiệu (fit scaler trên {len(tr)} trials)...", flush=True)\n    scaler = fit_fold_scaler(bundle.X, tr)\n    print(f"  [{fold[\'fold_key\']}] Chuẩn hóa xong! Bắt đầu huấn luyện {epochs} epoch...", flush=True)\n    train_loader = DataLoader(FoldDataset(bundle, tr, scaler), batch_size=batch_size, shuffle=True)\n    val_loader = DataLoader(FoldDataset(bundle, va, scaler), batch_size=batch_size)\n    test_loader = DataLoader(FoldDataset(bundle, te, scaler), batch_size=batch_size)\n    model = MSCNNBiGRU().to(device)\n    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))\n    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)\n    best_epoch, best_score = 0, -np.inf\n    epoch_losses = []\n    n_batches = len(train_loader)\n    for epoch in range(1, epochs + 1):\n        model.train()\n        batch_losses = []\n        for step, (x, y) in enumerate(train_loader, 1):\n            optimizer.zero_grad(set_to_none=True)\n            loss = loss_fn(model(x.to(device)), y.to(device))\n            loss.backward()\n            nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)\n            optimizer.step()\n            model.apply_max_norm()\n            batch_losses.append(loss.item())\n            if step % 20 == 0 or step == n_batches:\n                print(\n                    f"  [{fold[\'fold_key\']}] Epoch {epoch:2d}/{epochs} | Batch {step:3d}/{n_batches} | batch_loss={loss.item():.4f}",\n                    flush=True,\n                )\n        scheduler.step()\n        train_loss = float(np.mean(batch_losses))\n        epoch_losses.append(train_loss)\n        val_prob = _predict(model, val_loader, device)\n        metrics = compute_binary_metrics(bundle.y[va], val_prob, threshold=0.5)\n        score = metrics["roc_auc"] if np.isfinite(metrics["roc_auc"]) else metrics["balanced_accuracy"]\n        is_best = score > best_score + 1e-4\n        if is_best:\n            best_score, best_epoch = float(score), epoch\n            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n            best_val_prob = val_prob\n        marker = " ✓" if is_best else ""\n        no_improve = epoch - best_epoch\n        print(\n            f"  [{fold[\'fold_key\']}] Epoch {epoch:2d}/{epochs} | loss={train_loss:.4f}"\n            f" | val_auc={metrics[\'roc_auc\']:.4f}"\n            f" | val_ba={metrics[\'balanced_accuracy\']:.4f}"\n            f" | best_ep={best_epoch} (+{no_improve}){marker}",\n            flush=True,\n        )\n        if not is_best and no_improve >= patience:\n            print(f"  Early stop tại epoch {epoch} (patience={patience}).", flush=True)\n            break\n\n    if not best_epoch:\n        raise ValueError("Validation không cho metric hữu hạn")\n    model.load_state_dict(best_state)\n    threshold = float(find_optimal_threshold(bundle.y[va], best_val_prob, metric="balanced_accuracy"))\n    test_prob = _predict(model, test_loader, device)\n    elapsed = time.perf_counter() - start\n    fold_meta = {k: v for k, v in fold.items() if k not in ("train_idx", "val_idx", "test_idx")}\n    path = None\n    if checkpoint_dir is not None:\n        path = Path(checkpoint_dir) / (fold["fold_key"] + ".pt")\n        path.parent.mkdir(parents=True, exist_ok=True)\n        with NamedTemporaryFile(dir=path.parent, suffix=".tmp", delete=False) as f:\n            tmp = Path(f.name)\n        try:\n            torch.save(dict(model_name="ms_cnn_bigru", model_kwargs={}, model_state_dict=best_state,\n                            scaler_state_dict=scaler.state_dict(), best_threshold=threshold,\n                            best_epoch=best_epoch, best_val_metric=best_score, seed=seed,\n                            n_times=bundle.X.shape[2], trial_uid_sha256=bundle.trial_uid_sha256,\n                            fold_meta=fold_meta), tmp)\n            shutil.move(str(tmp), str(path))\n        finally:\n            tmp.unlink(missing_ok=True)\n\n    predictions = pd.DataFrame(dict(\n        trial_uid=bundle.metadata["trial_uid"].to_numpy()[te], subject_id=bundle.subject_ids[te],\n        protocol=fold["protocol"], fold=fold["fold"], y_true=bundle.y[te], y_prob=test_prob,\n        y_pred=(test_prob >= threshold).astype(np.int64), threshold=threshold,\n    ), columns=PREDICTION_COLUMNS)\n    return dict(\n        fold_metrics={**fold_meta, **compute_binary_metrics(bundle.y[te], test_prob, threshold=threshold),\n                      "pos_weight": pos_weight, "train_time_sec": elapsed},\n        val_metrics=compute_binary_metrics(bundle.y[va], best_val_prob, threshold=threshold),\n        predictions_df=predictions, best_threshold=threshold, best_epoch=best_epoch,\n        best_val_metric=best_score, train_time_sec=elapsed, checkpoint_path=str(path) if path else None,\n    )\n',
    'src/models/__init__.py': '"""Các kiến trúc Deep Learning cho phân loại single-trial EEG P300."""\n\nfrom src.models.ms_cnn_bigru import MSCNNBiGRU\n\n__all__ = ["EEGNetV4", "MSCNNBiGRU", "P300ATCNet"]\n\n\ndef __getattr__(name):\n    """Chỉ nạp kiến trúc DL2 khi người gọi yêu cầu model đó."""\n    if name == "EEGNetV4":\n        from .eegnet import EEGNetV4\n        return EEGNetV4\n    if name == "P300ATCNet":\n        from .p300_atcnet import P300ATCNet\n        return P300ATCNet\n    raise AttributeError(name)\n',
}


In [ ]:
#@title Chuẩn bị mã và thư mục kết quả DL1
import os
import shutil
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

DL1_MODE = "full"  #@param ["full", "pilot"]
DL1_RESULTS_DIR = (DRIVE_PROJECT_DIR / "results").resolve()

if "dl1_workspace" in globals():
    dl1_workspace.cleanup()
dl1_workspace = TemporaryDirectory(prefix="eeg-dl1-")
dl1_runtime = Path(dl1_workspace.name)
for name in ("src", "splits", "reports"):
    shutil.copytree(repo_dir / name, dl1_runtime / name, ignore=shutil.ignore_patterns("__pycache__"))
for relative, source in DL1_SNAPSHOT.items():
    target = dl1_runtime / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(source.encode("utf-8"))

def run_dl1(method):
    command = [
        sys.executable, "-u", str(dl1_runtime / "scripts/train_dl1.py"),
        "--method", method, "--mode", DL1_MODE,
        "--data-dir", str(processed_dir.resolve()), "--results-dir", str(DL1_RESULTS_DIR),
    ]
    with subprocess.Popen(
        command, cwd=dl1_runtime, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        encoding="utf-8", errors="replace", env={**os.environ, "PYTHONIOENCODING": "utf-8"},
    ) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
        except KeyboardInterrupt:
            process.terminate()
            raise
    if process.returncode:
        raise RuntimeError(f"DL1 {method.upper()} dừng với mã lỗi {process.returncode}.")

print(f"Kết quả và cache DL1 trên Drive: {DL1_RESULTS_DIR}")


### 1. Cross-subject 5-fold · MS-CNN-BiGRU

Chạy 5 fold `clean` rồi 5 fold `no_ica`. Hai biến thể dùng cùng phân nhóm subject, với cache và kết quả riêng.

In [ ]:
run_dl1("cs5")

### 2. Within-subject · MS-CNN-BiGRU

Chạy 5 fold trong từng subject trên dữ liệu `clean`, tổng cộng 195 fold ở chế độ `full`.

In [ ]:
run_dl1("within")

### 3. LOSO · MS-CNN-BiGRU

Lần lượt giữ riêng một subject để test trên dữ liệu `clean`, tổng cộng 39 fold ở chế độ `full`.

In [ ]:
run_dl1("loso")

### 3. Cross-subject 5-fold · MS-CNN-BiGRU

Chạy 5 fold `clean` rồi 5 fold `no_ica`. Hai biến thể dùng cùng phân nhóm subject, với cache và kết quả riêng.

In [ ]:
run_dl1("cs5")